# 🧠 Cross-Domain Generalization & Transferability Benchmark
### Multi-Center Out-of-Distribution Robustness Evaluation of 6 SOTA Paradigms on Brain MRI Lesion Delineation & Subtyping
---
**Target Journals:** *IEEE Transactions on Medical Imaging (TMI)* / *IEEE Journal of Biomedical and Health Informatics (JBHI)* / *Elsevier Computers in Biology and Medicine (CBM)*  
**Author / Lab:** Brain Tumor AI Research Initiative  
**Study Scope:** Standardized cross-cohort transferability evaluation across 3 independent multi-center cohorts using 18 pre-trained model checkpoints ($6 \text{ models} \times 3 \text{ cohorts}$).

---
### 🔬 Scientific Motivation & Theoretical Formulation:
In real-world clinical translation, deep learning systems trained at a single medical institution (Source Domain $\mathcal{D}_S$) frequently suffer catastrophic performance degradation when deployed on scans from external hospitals (Target Domain $\mathcal{D}_T$). This degradation arises from:
1. **Acquisition Shift:** Discrepancies in MRI field strengths (1.5T vs 3.0T), RF pulse sequences (T1-weighted contrast-enhanced vs T2/FLAIR), and manufacturer-specific artifacts (Siemens, GE, Philips).
2. **Demographic Shift:** Patient population variance in tumor grade distributions and anatomical morphologies.

#### Mathematical Definitions:
* **In-Domain Baseline Performance:**  
  $$\mathcal{M}_{\text{in}}(S) = \text{Metric}(\theta_S, \mathcal{D}_{S}^{\text{test}})$$
* **Cross-Domain Transfer Performance:**  
  $$\mathcal{M}_{\text{cross}}(S \rightarrow T) = \text{Metric}(\theta_S, \mathcal{D}_{T}^{\text{test}}) \quad (S \neq T)$$
* **Absolute Generalization Degradation Gap ($\Delta_{\text{Gen}}$):**  
  $$\Delta_{\text{Gen}}(S \rightarrow T) = \mathcal{M}_{\text{in}}(T) - \mathcal{M}_{\text{cross}}(S \rightarrow T)$$
* **Relative Generalization Retention Rate (RGRR):**  
  $$\text{RGRR}(S \rightarrow T) = \frac{\mathcal{M}_{\text{cross}}(S \rightarrow T)}{\mathcal{M}_{\text{in}}(T)} \times 100\%$$

---
### 🎯 Evaluated Architectures (All 6 SOTA Paradigms):
1. **Compound CNN:** Dual-Head U-Net++ (EfficientNet-B4)
2. **Pure Vision Transformer (ViT):** Dual-Head SegFormer (MiT-B4)
3. **Context Aggregation (ASPP):** Dual-Head DeepLabV3+ (ResNeXt-101)
4. **Multi-Axis Attention:** Dual-Head MA-Net (DenseNet-121)
5. **Feature Pyramid & SE:** Dual-Head FPN (SE-ResNeXt-50)
6. **Real-Time Edge Deployment:** YOLOv26-Seg (Ultralytics SOTA)

---
### 🏥 Evaluated Multi-Center Cohorts:
1. **Cheng Figshare:** 3,064 slices (Folds 1-4 Train/Val, Fold 5 Test = 766 slices | 3 Classes: Glioma, Meningioma, Pituitary)
2. **BRISC 2025:** 6,000 slices (Train/Val 5,000 slices, Test = 1,000 slices | 4 Classes: Glioma, Meningioma, Pituitary, No Tumor)
3. **Nickparvar MRI:** 7,045 slices (Train/Val 5,445 slices, Test = 1,600 slices | 4 Classes: Glioma, Meningioma, Pituitary, No Tumor)

---
### ⚖️ Fair Class Alignment Strategy:
* When evaluating between 3-class models (Cheng) and 4-class cohorts (BRISC / Nickparvar), metrics are calculated strictly on the **3 shared tumor classes (`glioma`, `meningioma`, `pituitary`)**.
* When evaluating between 4-class models and cohorts (BRISC $\leftrightarrow$ Nickparvar), full **4-class multi-task evaluation** is performed.


In [ ]:
# 1. Install & Verify Required Scientific & Deep Learning Libraries
!pip install -q segmentation-models-pytorch albumentations "ultralytics>=8.3.0" timm tabulate seaborn

import os
import sys
import time
import json
import glob
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A

print("=" * 80)
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Acceleration : {'ACTIVE ✅' if torch.cuda.is_available() else 'INACTIVE ❌'}")
if torch.cuda.is_available():
    print(f"Compute Device    : {torch.cuda.get_device_name(0)}")
    print(f"Dedicated VRAM    : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 80)


In [ ]:
# 2. Universal Dynamic Auto-Discovery of Datasets & Pretrained Model Checkpoints

print("=" * 85)
print("🔍 DISCOVERING MOUNTED DATASETS & CHECKPOINTS IN ENVIRONMENT...")
print("=" * 85)

# Step A: Locate Mounted Input Paths
mounted_items = []
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.count(os.sep) - '/kaggle/input'.count(os.sep) <= 3:
            mounted_items.append(root)
else:
    mounted_items = [
        'g:/My Drive/Brain Tumor Detection',
        'datasets',
        'training_outputs_cheng_figshare',
        'training_outputs_brisc2025',
        'training_outputs_nickparvar',
        '.'
    ]

def find_file_or_dir(target_name, keywords=None):
    # Direct candidate search
    candidates = [
        f"/kaggle/input/{target_name}",
        f"/kaggle/input/datasets/mdmahfujulkarim/{target_name}",
        f"g:/My Drive/Brain Tumor Detection/datasets/{target_name}",
        f"g:/My Drive/Brain Tumor Detection/{target_name}",
        f"datasets/{target_name}",
        f"./{target_name}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
            
    # Recursive search
    for root in mounted_items:
        check_p = os.path.join(root, target_name)
        if os.path.exists(check_p):
            return check_p
            
    if keywords:
        for root in mounted_items:
            norm = os.path.basename(root).lower().replace('-', '_').replace(' ', '_')
            if any(k in norm for k in keywords):
                return root
    return None

# Resolve 3 Dataset Roots
CHENG_DIR = None
BRISC_DIR = None
NICKPARVAR_DIR = None

# 1. Cheng Figshare
for cand in [
    '/kaggle/input/datasets/mdmahfujulkarim/cheng-figshare-last',
    '/kaggle/input/cheng-figshare-last',
    '/kaggle/input/cheng_figshare_last',
    'g:/My Drive/Brain Tumor Detection/datasets/cheng figshare last',
    'datasets/cheng figshare last'
]:
    if os.path.exists(cand) and os.path.exists(os.path.join(cand, 'cheng_figshare_metadata.csv')):
        CHENG_DIR = cand
        break
if CHENG_DIR is None:
    for m in mounted_items:
        if os.path.exists(os.path.join(m, 'cheng_figshare_metadata.csv')):
            CHENG_DIR = m
            break

# 2. BRISC 2025
for cand in [
    '/kaggle/input/datasets/mdmahfujulkarim/brisc2025',
    '/kaggle/input/brisc2025',
    'g:/My Drive/Brain Tumor Detection/datasets/brisc2025',
    'datasets/brisc2025'
]:
    if os.path.exists(cand) and os.path.exists(os.path.join(cand, 'manifest.csv')):
        BRISC_DIR = cand
        break
if BRISC_DIR is None:
    for m in mounted_items:
        if os.path.exists(os.path.join(m, 'manifest.csv')):
            BRISC_DIR = m
            break

# 3. Nickparvar MRI
for cand in [
    '/kaggle/input/datasets/mdmahfujulkarim/nickparvar-mri1/nickparvar_mri',
    '/kaggle/input/datasets/mdmahfujulkarim/nickparvar-mri1',
    '/kaggle/input/datasets/mdmahfujulkarim/nickparvar_mri1/nickparvar_mri',
    '/kaggle/input/datasets/mdmahfujulkarim/nickparvar_mri1',
    '/kaggle/input/datasets/mdmahfujulkarim/nickparvar-mri/nickparvar_mri',
    '/kaggle/input/datasets/mdmahfujulkarim/nickparvar-mri',
    '/kaggle/input/nickparvar_mri1/nickparvar_mri',
    '/kaggle/input/nickparvar_mri1',
    'g:/My Drive/Brain Tumor Detection/datasets/nickparvar_mri',
    'datasets/nickparvar_mri'
]:
    if os.path.exists(cand):
        if os.path.exists(os.path.join(cand, 'test')) or os.path.exists(os.path.join(cand, 'nickparvar_metadata.csv')):
            NICKPARVAR_DIR = cand
            break
        if os.path.exists(os.path.join(cand, 'nickparvar_mri', 'test')):
            NICKPARVAR_DIR = os.path.join(cand, 'nickparvar_mri')
            break

print("🎯 RESOLVED DATASET PATHS:")
print(f"  * Cheng Figshare : {CHENG_DIR}")
print(f"  * BRISC 2025     : {BRISC_DIR}")
print(f"  * Nickparvar MRI : {NICKPARVAR_DIR}")

# Step B: Auto-Discover Model Checkpoints (18 Checkpoints)
MODEL_NAMES = [
    "UNetPlusPlus-EfficientNetB4",
    "SegFormer-MiTB4",
    "DeepLabV3Plus-ResNeXt",
    "MANet-DenseNet121",
    "FPN-SEResNeXt50",
    "YOLOv26-Seg"
]
DATASET_KEYS = ["cheng_figshare", "brisc2025", "nickparvar_mri"]

CHECKPOINTS = {} # Key: (model_name, dataset_key) -> path

def search_checkpoint(m_name, d_key):
    ext = ".pt" if "YOLO" in m_name else ".pth"
    expected_filename = f"best_{m_name}_{d_key}{ext}"
    
    # Check known local / kaggle candidate folders
    candidate_folders = [
        # Kaggle input paths for trained-model-input dataset
        "/kaggle/input/trained-model-input",
        "/kaggle/input/trained_model_input",
        "/kaggle/input/datasets/mdmahfujulkarim/trained-model-input",
        "/kaggle/input/datasets/mdmahfujulkarim/trained_model_input",
        "/kaggle/input/datasets/mdmahfujulkarim/kaggle-outputs",
        "/kaggle/input/kaggle-outputs",
        f"/kaggle/input/training-outputs-{d_key}",
        f"/kaggle/input/training_outputs_{d_key}",
        f"/kaggle/input/brain-tumor-detection",
        f"/kaggle/input/brisc2025-training-notebook",
        f"/kaggle/input/cheng-figshare-training",
        "g:/My Drive/Brain Tumor Detection/trained model input",
        f"g:/My Drive/Brain Tumor Detection/training_outputs_{d_key}/benchmark_outputs",
        f"g:/My Drive/Brain Tumor Detection/training_outputs_{d_key}",
        "trained model input",
        "./trained model input",
        f"training_outputs_{d_key}/benchmark_outputs",
        f"./training_outputs_{d_key}/benchmark_outputs",
        "./benchmark_outputs",
        "."
    ]
    for folder in candidate_folders:
        direct_p = os.path.join(folder, expected_filename)
        if os.path.exists(direct_p):
            return direct_p
        # For YOLO, check nested run folder
        if "YOLO" in m_name:
            yolo_nested = os.path.join(folder, f"yolo26_{d_key}_run", "weights", "best.pt")
            if os.path.exists(yolo_nested):
                return yolo_nested
                
    # Search all mounted input paths recursively
    search_roots = ['/kaggle/input', 'g:/My Drive/Brain Tumor Detection', '.'] if os.path.exists('/kaggle/input') else ['g:/My Drive/Brain Tumor Detection', '.']
    for s_root in search_roots:
        if os.path.exists(s_root):
            for r, dirs, files in os.walk(s_root):
                for f in files:
                    if f == expected_filename:
                        return os.path.join(r, f)
                    if "YOLO" in m_name and "best.pt" in f and d_key in r:
                        return os.path.join(r, f)
    return None

checkpoint_status_table = []
for m in MODEL_NAMES:
    for d in DATASET_KEYS:
        ckpt_path = search_checkpoint(m, d)
        CHECKPOINTS[(m, d)] = ckpt_path
        status_str = "FOUND ✅" if ckpt_path is not None else "MISSING ❌"
        short_p = os.path.basename(ckpt_path) if ckpt_path else "None"
        checkpoint_status_table.append([m, d, status_str, short_p])

print("\n" + "=" * 85)
print("📊 CHECKPOINT AUTO-DISCOVERY AUDIT (18 TARGET WEIGHTS):")
print("=" * 85)
print(tabulate(checkpoint_status_table, headers=["Model Name", "Source Cohort", "Status", "Resolved Filename"], tablefmt="fancy_grid"))


In [ ]:
# 3. Standardized Preprocessing Pipeline & Hold-Out Test DataLoaders

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMAGE_SIZE    = 256
BATCH_SIZE    = 32

def crop_brain_contour(image: np.ndarray, mask: np.ndarray = None, pad_ratio: float = 0.05):
    """Skull-stripping contour crop with 5% safety margin to protect boundary meningiomas."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image.copy()
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    h_img, w_img = gray.shape
    if contours:
        c = max(contours, key=cv2.contourArea)
        if cv2.contourArea(c) > (h_img * w_img * 0.05):
            x, y, w, h = cv2.boundingRect(c)
            pad_w, pad_h = int(w * pad_ratio), int(h * pad_ratio)
            x1, y1 = max(0, x - pad_w), max(0, y - pad_h)
            x2, y2 = min(w_img, x + w + pad_w), min(h_img, y + h + pad_h)
            cropped_img = image[y1:y2, x1:x2]
            cropped_mask = mask[y1:y2, x1:x2] if mask is not None else None
            return cropped_img, cropped_mask
    return image, mask

def apply_clahe(image: np.ndarray, clip_limit: float = 2.0, tile_grid_size=(8, 8)) -> np.ndarray:
    """Contrast Limited Adaptive Histogram Equalization for enhanced soft-tissue contrast."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    if len(image.shape) == 2:
        return clahe.apply(image)
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    cl = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB)

class UniversalTestDataset(Dataset):
    """Universal Hold-out Test Loader across Cheng, BRISC, and Nickparvar cohorts."""
    def __init__(self, dataset_name, base_dir, image_size=256):
        self.dataset_name = dataset_name
        self.base_dir = base_dir
        self.image_size = image_size
        self.samples = []
        
        # Mappings: 0: glioma, 1: meningioma, 2: pituitary, 3: no_tumor
        if dataset_name == 'cheng_figshare':
            csv_path = os.path.join(base_dir, 'cheng_figshare_metadata.csv')
            df = pd.read_csv(csv_path)
            test_df = df[df['split'] == 'test'].reset_index(drop=True)
            class_map = {'glioma': 0, 'meningioma': 1, 'pituitary': 2}
            for _, row in test_df.iterrows():
                img_rel = str(row['image_path']).replace(chr(92), '/')
                mask_rel = str(row['mask_path']).replace(chr(92), '/')
                self.samples.append({
                    'img_path': os.path.join(base_dir, img_rel),
                    'mask_path': os.path.join(base_dir, mask_rel),
                    'label': class_map[row['class_name']],
                    'class_name': row['class_name'],
                    'has_mask': True
                })
        elif dataset_name == 'brisc2025':
            csv_path = os.path.join(base_dir, 'manifest.csv')
            df = pd.read_csv(csv_path)
            seg_masks = df[(df['task'] == 'segmentation') & (df['is_mask'] == True)]
            mask_lookup = {row['filename']: str(row['relative_path']).replace(chr(92), '/') for _, row in seg_masks.iterrows()}
            test_df = df[(df['task'] == 'classification') & (df['split'] == 'test')].reset_index(drop=True)
            class_map = {'glioma': 0, 'meningioma': 1, 'pituitary': 2, 'no_tumor': 3}
            for _, row in test_df.iterrows():
                fname = row['filename']
                rel_p = str(row['relative_path']).replace(chr(92), '/')
                m_path = os.path.join(base_dir, mask_lookup[fname]) if fname in mask_lookup else None
                self.samples.append({
                    'img_path': os.path.join(base_dir, rel_p),
                    'mask_path': m_path,
                    'label': class_map[row['tumor_label']],
                    'class_name': row['tumor_label'],
                    'has_mask': (m_path is not None) or (row['tumor_label'] == 'no_tumor')
                })
        else: # nickparvar_mri
            test_dir = os.path.join(base_dir, 'test')
            class_map = {'glioma': 0, 'meningioma': 1, 'pituitary': 2, 'notumor': 3}
            for c in ['glioma', 'meningioma', 'pituitary', 'notumor']:
                c_dir = os.path.join(test_dir, c)
                if os.path.exists(c_dir):
                    for fname in sorted(os.listdir(c_dir)):
                        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                            f_path = os.path.join(c_dir, fname)
                            base_name = os.path.splitext(fname)[0]
                            m_path = os.path.join(base_dir, "masks", "test", c, f"{base_name}_mask.png")
                            has_m = os.path.exists(m_path) or (c == 'notumor')
                            self.samples.append({
                                'img_path': f_path,
                                'mask_path': m_path if os.path.exists(m_path) else None,
                                'label': class_map[c],
                                'class_name': 'no_tumor' if c == 'notumor' else c,
                                'has_mask': has_m
                            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = cv2.imread(item['img_path'], cv2.IMREAD_GRAYSCALE)
        if image is None:
            image = np.zeros((self.image_size, self.image_size), dtype=np.uint8)
            
        mask = None
        if item['mask_path'] and os.path.exists(item['mask_path']):
            mask = cv2.imread(item['mask_path'], cv2.IMREAD_GRAYSCALE)
        elif item['class_name'] in ['no_tumor', 'notumor']:
            mask = np.zeros_like(image, dtype=np.uint8)
            
        # Skull crop & CLAHE
        image, mask = crop_brain_contour(image, mask)
        image = apply_clahe(image)
        
        # Resize
        image = cv2.resize(image, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        if mask is not None:
            mask = cv2.resize(mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)
            
        # Convert to 3-channel normalized tensor
        img_rgb = np.stack([image, image, image], axis=-1).astype(np.float32) / 255.0
        img_norm = (img_rgb - IMAGENET_MEAN) / IMAGENET_STD
        img_tensor = torch.tensor(img_norm.transpose(2, 0, 1), dtype=torch.float32)
        mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        
        return {
            'image': img_tensor,
            'mask': mask_tensor,
            'label': torch.tensor(item['label'], dtype=torch.long),
            'has_mask': torch.tensor(item['has_mask'], dtype=torch.bool),
            'class_name': item['class_name']
        }

# Instantiate Hold-out Test DataLoaders
test_loaders = {}
if CHENG_DIR:
    cheng_test_ds = UniversalTestDataset('cheng_figshare', CHENG_DIR, image_size=IMAGE_SIZE)
    test_loaders['cheng_figshare'] = DataLoader(cheng_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    print(f"✅ Cheng Figshare Test DataLoader Ready : {len(cheng_test_ds):,} slices ({len(test_loaders['cheng_figshare'])} batches)")

if BRISC_DIR:
    brisc_test_ds = UniversalTestDataset('brisc2025', BRISC_DIR, image_size=IMAGE_SIZE)
    test_loaders['brisc2025'] = DataLoader(brisc_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    print(f"✅ BRISC 2025 Test DataLoader Ready     : {len(brisc_test_ds):,} slices ({len(test_loaders['brisc2025'])} batches)")

if NICKPARVAR_DIR:
    nick_test_ds = UniversalTestDataset('nickparvar_mri', NICKPARVAR_DIR, image_size=IMAGE_SIZE)
    test_loaders['nickparvar_mri'] = DataLoader(nick_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    print(f"✅ Nickparvar MRI Test DataLoader Ready : {len(nick_test_ds):,} slices ({len(test_loaders['nickparvar_mri'])} batches)")


In [ ]:
# 4. Multi-Task SOTA Architectures & Checkpoint Loader (Models 1 to 5)

import segmentation_models_pytorch as smp

class DualHeadUNetPlusPlus(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name="efficientnet-b4",
            encoder_weights=None,
            in_channels=3,
            classes=1,
            aux_params=dict(pooling='avg', dropout=0.3, classes=num_classes)
        )
    def forward(self, x):
        mask_logits, class_logits = self.model(x)
        return torch.sigmoid(mask_logits), class_logits

class DualHeadSegFormer(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = smp.Unet(
            encoder_name="mit_b4",
            encoder_weights=None,
            in_channels=3,
            classes=1,
            aux_params=dict(pooling='avg', dropout=0.3, classes=num_classes)
        )
    def forward(self, x):
        mask_logits, class_logits = self.model(x)
        return torch.sigmoid(mask_logits), class_logits

class DualHeadDeepLabV3Plus(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = smp.DeepLabV3Plus(
            encoder_name="resnext101_32x8d",
            encoder_weights=None,
            in_channels=3,
            classes=1,
            aux_params=dict(pooling='avg', dropout=0.3, classes=num_classes)
        )
    def forward(self, x):
        mask_logits, class_logits = self.model(x)
        return torch.sigmoid(mask_logits), class_logits

class DualHeadMANet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = smp.MAnet(
            encoder_name="densenet121",
            encoder_weights=None,
            in_channels=3,
            classes=1,
            aux_params=dict(pooling='avg', dropout=0.3, classes=num_classes)
        )
    def forward(self, x):
        mask_logits, class_logits = self.model(x)
        return torch.sigmoid(mask_logits), class_logits

class DualHeadFPN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.model = smp.FPN(
            encoder_name="se_resnext50_32x4d",
            encoder_weights=None,
            in_channels=3,
            classes=1,
            aux_params=dict(pooling='avg', dropout=0.3, classes=num_classes)
        )
    def forward(self, x):
        mask_logits, class_logits = self.model(x)
        return torch.sigmoid(mask_logits), class_logits

PYTORCH_FACTORY = {
    "UNetPlusPlus-EfficientNetB4": DualHeadUNetPlusPlus,
    "SegFormer-MiTB4":             DualHeadSegFormer,
    "DeepLabV3Plus-ResNeXt":       DualHeadDeepLabV3Plus,
    "MANet-DenseNet121":           DualHeadMANet,
    "FPN-SEResNeXt50":             DualHeadFPN
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_pytorch_model(model_name, source_cohort, ckpt_path):
    num_classes = 3 if source_cohort == 'cheng_figshare' else 4
    model_cls = PYTORCH_FACTORY[model_name]
    model = model_cls(num_classes=num_classes).to(device)
    
    state_dict = torch.load(ckpt_path, map_location=device)
    # Handle state_dict wrapper if present
    if 'state_dict' in state_dict:
        state_dict = state_dict['state_dict']
    model.load_state_dict(state_dict)
    model.eval()
    return model


In [ ]:
# 5. Scientific Metric Calculation & Fair Class Alignment Engine

from sklearn.metrics import f1_score, accuracy_score

def compute_batch_dice(pred_probs, target_masks, threshold=0.5, smooth=1e-6):
    preds = (pred_probs > threshold).float()
    intersection = (preds * target_masks).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + target_masks.sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice

def compute_batch_iou(pred_probs, target_masks, threshold=0.5, smooth=1e-6):
    preds = (pred_probs > threshold).float()
    intersection = (preds * target_masks).sum(dim=(1, 2, 3))
    total = preds.sum(dim=(1, 2, 3)) + target_masks.sum(dim=(1, 2, 3))
    union = total - intersection
    iou = (intersection + smooth) / (union + smooth)
    return iou

@torch.no_grad()
def evaluate_pytorch_cross_domain(model, source_cohort, target_cohort, dataloader):
    """
    Fair Multi-Task Cross-Domain Evaluation:
    - If source is 3-class (Cheng) & target is 4-class: Evaluates on Common-3 (glioma, meningioma, pituitary)
    - If source is 4-class & target is 3-class: Restricts predicted logits to the 3 shared classes
    - If both 4-class (BRISC <-> Nickparvar): Full 4-class evaluation
    """
    model.eval()
    all_dices = []
    all_ious = []
    all_preds = []
    all_targets = []
    
    source_is_3 = (source_cohort == 'cheng_figshare')
    target_is_3 = (target_cohort == 'cheng_figshare')
    
    for batch in dataloader:
        images = batch['image'].to(device)
        masks  = batch['mask'].to(device)
        labels = batch['label'].to(device)
        has_mask = batch['has_mask'].to(device)
        
        pred_mask_probs, pred_class_logits = model(images)
        
        # --- SEGMENTATION EVALUATION ---
        if has_mask.sum() > 0:
            sub_preds = pred_mask_probs[has_mask]
            sub_masks = masks[has_mask]
            dices = compute_batch_dice(sub_preds, sub_masks)
            ious  = compute_batch_iou(sub_preds, sub_masks)
            all_dices.extend(dices.cpu().numpy().tolist())
            all_ious.extend(ious.cpu().numpy().tolist())
            
        # --- CLASSIFICATION EVALUATION & CLASS ALIGNMENT ---
        if source_is_3 and not target_is_3:
            # Source model knows 0, 1, 2. Target has 0, 1, 2, 3 (no_tumor).
            # Evaluate on Common-3 subset (tumor cases):
            valid_idx = (labels < 3)
            if valid_idx.sum() > 0:
                cls_preds = torch.argmax(pred_class_logits[valid_idx], dim=1)
                all_preds.extend(cls_preds.cpu().numpy().tolist())
                all_targets.extend(labels[valid_idx].cpu().numpy().tolist())
        elif not source_is_3 and target_is_3:
            # Source model knows 0, 1, 2, 3. Target only has 0, 1, 2.
            # Mask out logit for class 3 (no_tumor) to evaluate fair choice among 3 tumor classes:
            restricted_logits = pred_class_logits[:, :3]
            cls_preds = torch.argmax(restricted_logits, dim=1)
            all_preds.extend(cls_preds.cpu().numpy().tolist())
            all_targets.extend(labels.cpu().numpy().tolist())
        else:
            # Standard identical domain class counts (3 vs 3 or 4 vs 4)
            cls_preds = torch.argmax(pred_class_logits, dim=1)
            all_preds.extend(cls_preds.cpu().numpy().tolist())
            all_targets.extend(labels.cpu().numpy().tolist())
            
    mean_dice = float(np.mean(all_dices)) if len(all_dices) > 0 else 0.0
    mean_iou  = float(np.mean(all_ious)) if len(all_ious) > 0 else 0.0
    acc       = float(accuracy_score(all_targets, all_preds) * 100.0) if len(all_targets) > 0 else 0.0
    f1        = float(f1_score(all_targets, all_preds, average='macro') * 100.0) if len(all_targets) > 0 else 0.0
    
    return {
        'dice': round(mean_dice, 4),
        'iou':  round(mean_iou, 4),
        'acc':  round(acc, 2),
        'f1':   round(f1, 2)
    }


In [ ]:
# 6. YOLOv26-Seg Instance Segmentation Cross-Domain Evaluator

from ultralytics import YOLO

def evaluate_yolo_cross_domain(yolo_weight_path, source_cohort, target_cohort, dataloader):
    """
    Evaluates trained YOLOv26-Seg model on target test DataLoader.
    Extracts polygon segmentation overlap and classification accuracy.
    """
    if not yolo_weight_path or not os.path.exists(yolo_weight_path):
        return {'dice': 0.0, 'iou': 0.0, 'acc': 0.0, 'f1': 0.0}
        
    model = YOLO(yolo_weight_path)
    all_dices = []
    all_ious = []
    all_preds = []
    all_targets = []
    
    source_is_3 = (source_cohort == 'cheng_figshare')
    target_is_3 = (target_cohort == 'cheng_figshare')
    
    for batch in dataloader:
        images = batch['image'].numpy() # [B, 3, H, W]
        masks  = batch['mask'].squeeze(1).numpy() # [B, H, W]
        labels = batch['label'].numpy()
        has_mask = batch['has_mask'].numpy()
        
        # Denormalize to uint8 BGR for YOLO forward pass
        b = images.shape[0]
        for i in range(b):
            img_np = images[i].transpose(1, 2, 0)
            img_np = np.clip(img_np * IMAGENET_STD + IMAGENET_MEAN, 0, 1)
            img_bgr = (img_np * 255).astype(np.uint8)
            img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_RGB2BGR)
            
            target_lbl = labels[i]
            # Alignment filter for classification
            if source_is_3 and not target_is_3 and target_lbl == 3:
                continue # Skip healthy brain for 3-class source YOLO
                
            results = model.predict(img_bgr, imgsz=IMAGE_SIZE, verbose=False, device=0 if torch.cuda.is_available() else 'cpu')
            res = results[0]
            
            # Predict class & mask
            pred_class = 0
            pred_mask_bin = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
            
            if res.boxes is not None and len(res.boxes) > 0:
                pred_class = int(res.boxes.cls[0].item())
                if res.masks is not None and len(res.masks) > 0:
                    pred_poly_mask = res.masks.data[0].cpu().numpy()
                    pred_mask_bin = cv2.resize(pred_poly_mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
                    pred_mask_bin = (pred_mask_bin > 0.5).astype(np.float32)
            else:
                pred_class = 3 if not source_is_3 else 0
                
            # Alignment for 4-to-3 transfer:
            if not source_is_3 and target_is_3 and pred_class == 3:
                pred_class = 0 # fallback to most common tumor
                
            all_preds.append(pred_class)
            all_targets.append(target_lbl)
            
            if has_mask[i]:
                gt_m = masks[i]
                inter = np.sum(pred_mask_bin * gt_m)
                union = np.sum(pred_mask_bin) + np.sum(gt_m)
                d = (2.0 * inter + 1e-6) / (union + 1e-6)
                iou = (inter + 1e-6) / (union - inter + 1e-6)
                all_dices.append(d)
                all_ious.append(iou)
                
    mean_dice = float(np.mean(all_dices)) if len(all_dices) > 0 else 0.0
    mean_iou  = float(np.mean(all_ious)) if len(all_ious) > 0 else 0.0
    acc       = float(accuracy_score(all_targets, all_preds) * 100.0) if len(all_targets) > 0 else 0.0
    f1        = float(f1_score(all_targets, all_preds, average='macro') * 100.0) if len(all_targets) > 0 else 0.0
    
    return {
        'dice': round(mean_dice, 4),
        'iou':  round(mean_iou, 4),
        'acc':  round(acc, 2),
        'f1':   round(f1, 2)
    }


In [ ]:
# 7. Master Cross-Domain Execution Loop Across All 6 SOTA Models

print("=" * 90)
print("🚀 INITIATING MASTER CROSS-DOMAIN TRANSFER BENCHMARK (6 MODELS x 3 SOURCES x 3 TARGETS)")
print("=" * 90)

all_results_records = []
start_total_time = time.time()

for model_name in MODEL_NAMES:
    print(f"\n{'=' * 90}")
    print(f"🔹 EVALUATING ARCHITECTURE: [{model_name}]")
    print(f"{'=' * 90}")
    
    for s_cohort in DATASET_KEYS:
        ckpt_path = CHECKPOINTS.get((model_name, s_cohort))
        if not ckpt_path or not os.path.exists(ckpt_path):
            print(f"  ⚠️ Skipping Source [{s_cohort}]: Checkpoint not found.")
            continue
            
        print(f"\n  Source Model Checkpoint : {s_cohort.upper()} ({os.path.basename(ckpt_path)})")
        
        # Load model instance
        if "YOLO" in model_name:
            py_model = None
        else:
            py_model = load_pytorch_model(model_name, s_cohort, ckpt_path)
            
        for t_cohort in DATASET_KEYS:
            if t_cohort not in test_loaders:
                continue
                
            t_loader = test_loaders[t_cohort]
            is_in_domain = (s_cohort == t_cohort)
            eval_type = "In-Domain" if is_in_domain else "Cross-Domain"
            
            t0 = time.time()
            if "YOLO" in model_name:
                metrics = evaluate_yolo_cross_domain(ckpt_path, s_cohort, t_cohort, t_loader)
            else:
                metrics = evaluate_pytorch_cross_domain(py_model, s_cohort, t_cohort, t_loader)
            elapsed = time.time() - t0
            
            record = {
                'Model Name': model_name,
                'Source Domain (Train)': s_cohort,
                'Target Domain (Test)': t_cohort,
                'Evaluation Type': eval_type,
                'Dice Score': metrics['dice'],
                'Mean IoU': metrics['iou'],
                'Accuracy (%)': metrics['acc'],
                'Macro F1 (%)': metrics['f1'],
                'Eval Time (s)': round(elapsed, 2)
            }
            all_results_records.append(record)
            
            tag = "🎯 [IN-DOMAIN] " if is_in_domain else "🔄 [TRANSFER]  "
            print(f"    {tag} {s_cohort:>15} -> {t_cohort:<15} | Dice: {metrics['dice']:.4f} | IoU: {metrics['iou']:.4f} | Acc: {metrics['acc']:5.2f}% | F1: {metrics['f1']:5.2f}% ({elapsed:.1f}s)")

total_mins = (time.time() - start_total_time) / 60.0
print(f"\n✅ Completed Master Benchmark Across All 6 Models in {total_mins:.2f} minutes.")


In [ ]:
# 8. Statistical Matrix Aggregation & Generalization Drop (Delta_Gen) Computation

results_df = pd.DataFrame(all_results_records)
os.makedirs("cross_domain_outputs", exist_ok=True)
results_df.to_csv("cross_domain_outputs/cross_domain_raw_results_all_models.csv", index=False)

# Compute 3x3 Matrices per Model
matrices_dice = {}
matrices_acc  = {}
generalization_summary = []

for m in MODEL_NAMES:
    sub_df = results_df[results_df['Model Name'] == m]
    if len(sub_df) == 0:
        continue
        
    piv_dice = sub_df.pivot(index='Source Domain (Train)', columns='Target Domain (Test)', values='Dice Score')
    piv_acc  = sub_df.pivot(index='Source Domain (Train)', columns='Target Domain (Test)', values='Accuracy (%)')
    
    matrices_dice[m] = piv_dice
    matrices_acc[m]  = piv_acc
    
    # Calculate In-Domain Avg, Cross-Domain Avg, and Delta Drop
    in_domain_dices = []
    cross_domain_dices = []
    
    for s in DATASET_KEYS:
        for t in DATASET_KEYS:
            if s in piv_dice.index and t in piv_dice.columns:
                val = piv_dice.loc[s, t]
                if s == t:
                    in_domain_dices.append(val)
                else:
                    cross_domain_dices.append(val)
                    
    avg_in = np.mean(in_domain_dices) if in_domain_dices else 0.0
    avg_cross = np.mean(cross_domain_dices) if cross_domain_dices else 0.0
    delta_drop = avg_in - avg_cross
    retention_rate = (avg_cross / avg_in * 100.0) if avg_in > 0 else 0.0
    
    generalization_summary.append({
        'Model Name': m,
        'In-Domain Mean Dice': round(avg_in, 4),
        'Cross-Domain Mean Dice': round(avg_cross, 4),
        'Delta Gen Drop (Δ)': round(delta_drop, 4),
        'Retention Rate (%)': round(retention_rate, 2)
    })

gen_summary_df = pd.DataFrame(generalization_summary).sort_values(by='Delta Gen Drop (Δ)', ascending=True).reset_index(drop=True)
gen_summary_df.to_csv("cross_domain_outputs/model_generalizability_ranking.csv", index=False)

print("=" * 85)
print("🏆 GENERALIZABILITY RANKING ACROSS ALL 6 MODELS (LOWEST DROP = MOST ROBUST):")
print("=" * 85)
print(tabulate(gen_summary_df, headers='keys', tablefmt='fancy_grid', showindex=True))


In [ ]:
# 9. Publication-Grade Visualization: 3x3 Transfer Heatmaps for All 6 Models

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(2, 3, figsize=(18, 11), dpi=300)
axes = axes.flatten()

cohort_labels = ['Cheng', 'BRISC', 'Nickparvar']

for idx, m in enumerate(MODEL_NAMES):
    ax = axes[idx]
    if m in matrices_dice:
        mat = matrices_dice[m].reindex(index=DATASET_KEYS, columns=DATASET_KEYS)
        mat.index = cohort_labels
        mat.columns = cohort_labels
        
        sns.heatmap(
            mat,
            annot=True,
            fmt=".4f",
            cmap="Blues",
            cbar=True,
            vmin=0.60,
            vmax=0.98,
            linewidths=1.5,
            linecolor='white',
            ax=ax,
            annot_kws={"size": 11, "weight": "bold"}
        )
        ax.set_title(f"({chr(97+idx)}) {m}", fontsize=13, fontweight='bold', pad=10)
        ax.set_xlabel("Target Evaluation Domain", fontsize=10, fontweight='bold')
        ax.set_ylabel("Source Training Domain", fontsize=10, fontweight='bold')

plt.suptitle("Cross-Domain Lesion Delineation Generalization Matrices (Dice Score) Across 6 Deep Learning Paradigms", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
heatmap_dice_path = "cross_domain_outputs/fig_cross_domain_dice_heatmaps_all_models.png"
plt.savefig(heatmap_dice_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ High-Resolution Dice Heatmaps saved to: {heatmap_dice_path}")

# Figure 2: Classification Accuracy Heatmaps
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 11), dpi=300)
axes2 = axes2.flatten()

for idx, m in enumerate(MODEL_NAMES):
    ax = axes2[idx]
    if m in matrices_acc:
        mat = matrices_acc[m].reindex(index=DATASET_KEYS, columns=DATASET_KEYS)
        mat.index = cohort_labels
        mat.columns = cohort_labels
        
        sns.heatmap(
            mat,
            annot=True,
            fmt=".2f",
            cmap="Greens",
            cbar=True,
            vmin=70.0,
            vmax=100.0,
            linewidths=1.5,
            linecolor='white',
            ax=ax,
            annot_kws={"size": 11, "weight": "bold"}
        )
        ax.set_title(f"({chr(97+idx)}) {m} - Accuracy (%)", fontsize=13, fontweight='bold', pad=10)
        ax.set_xlabel("Target Evaluation Domain", fontsize=10, fontweight='bold')
        ax.set_ylabel("Source Training Domain", fontsize=10, fontweight='bold')

plt.suptitle("Cross-Domain Histopathological Subtyping Accuracy (%) Across 6 Deep Learning Paradigms", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
heatmap_acc_path = "cross_domain_outputs/fig_cross_domain_accuracy_heatmaps_all_models.png"
plt.savefig(heatmap_acc_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ High-Resolution Accuracy Heatmaps saved to: {heatmap_acc_path}")


In [ ]:
# 10. Publication-Ready LaTeX Tables & Comprehensive Zip Packaging

latex_lines = [
    r"% =========================================================================",
    r"% IEEE / Elsevier Master Cross-Domain Generalization Benchmark Table",
    r"% =========================================================================",
    r"\begin{table*}[t]",
    r"\centering",
    r"\caption{Standardized Multi-Center Cross-Domain Transfer Performance Across 6 Deep Learning Paradigms. In-domain baseline results are highlighted in bold. $\Delta_{\text{Gen}}$ represents the absolute generalization drop under distribution shift.}",
    r"\label{tab:cross_domain_master}",
    r"\resizebox{\textwidth}{!}{%",
    r"\begin{tabular}{llccccccc}",
    r"\toprule",
    r"\textbf{Model Architecture} & \textbf{Paradigm} & \textbf{Source Domain} & \textbf{Target Domain} & \textbf{Transfer Type} & \textbf{Dice Score} & \textbf{Mean IoU} & \textbf{Accuracy (\%)} & \textbf{Macro F1 (\%)} \\",
    r"\midrule"
]

for _, row in results_df.iterrows():
    is_in = (row['Evaluation Type'] == 'In-Domain')
    d_str = f"\\textbf{{{row['Dice Score']:.4f}}}" if is_in else f"{row['Dice Score']:.4f}"
    a_str = f"\\textbf{{{row['Accuracy (%)']:.2f}}}" if is_in else f"{row['Accuracy (%)']:.2f}"
    m_name = str(row['Model Name']).replace('_', '-')
    s_dom  = str(row['Source Domain (Train)']).replace('_', ' ')
    t_dom  = str(row['Target Domain (Test)']).replace('_', ' ')
    e_type = str(row['Evaluation Type'])
    iou_s  = f"{row['Mean IoU']:.4f}"
    f1_s   = f"{row['Macro F1 (%)']:.2f}"
    latex_lines.append(f"{m_name} & Multi-Task & {s_dom} & {t_dom} & {e_type} & {d_str} & {iou_s} & {a_str} & {f1_s} \\\\")

latex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}%",
    r"}",
    r"\end{table*}"
])

latex_content = "\n".join(latex_lines)

with open("cross_domain_outputs/latex_cross_domain_table.tex", "w", encoding="utf-8") as f_tex:
    f_tex.write(latex_content)

# Packaging Everything into a Single Downloadable Archive
zip_archive_path = "cross_domain_all_models_results.zip"
if os.path.exists(zip_archive_path):
    os.remove(zip_archive_path)

shutil.make_archive("cross_domain_all_models_results", 'zip', "cross_domain_outputs")
print("=" * 85)
print("📦 ALL RESEARCH DELIVERABLES PACKAGED SUCCESSFULLY!")
print(f"   -> Destination Archive: {os.path.abspath(zip_archive_path)}")
print(f"   -> Archive Size: {os.path.getsize(zip_archive_path) / (1024**2):.2f} MB")
print("=" * 85)
